# Generare musica

Il codice del capitolo [«Generare musica»](https://book.paithon.it/main/GenerazioneMusica/overview.html), *Paithon Book*.

Le celle sono quelle del libro, nell'ordine in cui compaiono: il testo che le spiega sta nelle pagine, qui c'è solo la parte da eseguire e da rompere.

Generato da `scripts/genera-notebook.py`: le correzioni vanno fatte nelle pagine del libro, non qui.


> **Verificato il 2026-07-25** con torch 2.13.0, numpy 2.4.6, pandas 3.0.5, scikit-learn 1.9.0, transformers 5.14.1, diffusers 0.39.0, librosa 0.11.0, torch-geometric 2.8.0.post1. Tutte le celle di questo notebook sono state eseguite senza errori con quelle versioni; le librerie si muovono, e se qualcosa qui non gira piu' e' un errore del libro: [segnalalo](https://github.com/paithon-it/paithonbook/issues).


In [ ]:
# Su Colab quasi tutto c'è già; questa riga serve altrove.
%pip install -q numpy

In [ ]:
# Mostra il valore di ogni riga, come i commenti «# ->» del libro.
try:
    from IPython.core.interactiveshell import InteractiveShell
    InteractiveShell.ast_node_interactivity = 'all'
except ImportError:      # fuori da IPython non serve e non c'è
    pass

## Generare musica

[Leggi la pagina](https://book.paithon.it/main/GenerazioneMusica/overview.html)


### Che cosa ha in più una canzone


In [ ]:
# Quattro minuti di canzone stereo contro il testo che la descrive.
D = 240          # durata in secondi
fs = 44100       # frequenza di campionamento (Hz)
canali = 2       # stereo

numeri = D * fs * canali

strofa = """Sulla riva della notte
una luce piccola resta
e il mare ripete piano
il nome che non ti ho detto""".replace("\n", " ")
strofe = 6
caratteri = len(strofa) * strofe


def con_punti(x):
    return f"{x:,}".replace(",", ".")


print("numeri al secondo (stereo):", con_punti(fs * canali))
print("numeri nell'onda di 240 s: ", con_punti(numeri))
print("caratteri di", strofe, "strofe:     ", con_punti(caratteri))
print("numeri per ogni carattere: ", con_punti(numeri // caratteri))

# lo stesso rapporto per un testo corto e per uno lungo
for c in (200, 3000):
    print(f"con {con_punti(c):>5} caratteri:       ", con_punti(numeri // c))

### L'allineamento fra testo e tempo


In [ ]:
from itertools import combinations
from math import comb

# Le sillabe di una strofa, contate senza sinalefe.
strofa = ("Sul la ri va del la not te u na lu ce pic co la re sta "
          "e il ma re ri pe te pia no il no me che non ti ho det to")
sillabe = strofa.split()
strofe = 6
L = len(sillabe) * strofe     # sillabe dell'intera canzone

r = 50                        # frame al secondo
D = 240                       # secondi
n = r * D                     # frame della canzone


def con_punti(x):
    return f"{x:,}".replace(",", ".")


# il caso più piccolo: 2 sillabe in 4 istanti, contati da 0
coppie = list(combinations(range(4), 2))
print("modi di porre 2 sillabe in 4 istanti:", len(coppie))
print(coppie)

# la prima riga: otto sillabe distese su otto secondi
riga = sillabe[:8]
modi_riga = comb(r * 8, len(riga))
print("sillabe per strofa:", len(sillabe))
print("prima riga:", " ".join(riga))
print("modi di porre 8 sillabe in 8 s:", con_punti(modi_riga))

# la canzone intera: L istanti d'inizio crescenti fra n frame
modi = comb(n, L)
print("sillabe della canzone:", L, " frame:", con_punti(n))
print("cifre del numero di allineamenti:", len(str(modi)))

## La voce che canta: dalla partitura al suono

[Leggi la pagina](https://book.paithon.it/main/GenerazioneMusica/la-voce-che-canta.html)


### Perché cantare non è parlare


In [ ]:
import numpy as np

def frequenza(m):
    """Frequenza fondamentale (Hz) della nota MIDI m: la nota 69 e' 440 Hz."""
    return 440.0 * 2.0 ** ((m - 69) / 12)

# La nota scritta: il MI sopra il do centrale (numero MIDI 64), tenuta 1,28 s.
m, durata = 64, 1.28
fm = frequenza(m)
print(f"nota MIDI {m}: f_m = {fm:.2f} Hz")

# Dal la (69) al MI (64), un tasto alla volta: ognuno moltiplica per 2^(-1/12).
scala = [round(frequenza(n)) for n in range(69, m - 1, -1)]
print(f"dal la al MI: {scala}, fattore {2 ** (-1 / 12):.3f} a tasto")

# Cento cent fanno un semitono, 1200 cent un'ottava: la scala e' logaritmica.
ottava = frequenza(m + 12) / fm
cent = 1200 * np.log2(ottava)
print(f"dodici semitoni: rapporto {ottava:.1f}, {cent:.0f} cent")

# Vibrato: oscillazione in cent attorno alla nota (A e f_v: valori di esempio).
A, fv = 50.0, 5.5                     # profondita' (cent), frequenza (Hz)
tau = np.arange(0.0, durata, 0.001)   # tempo in secondi, un campione al ms
f0_tau = fm * 2.0 ** (A * np.sin(2 * np.pi * fv * tau) / 1200)

print(f"f0(tau) fra {f0_tau.min():.2f} e {f0_tau.max():.2f} Hz")
sopra, sotto = f0_tau.max() - fm, f0_tau.min() - fm
print(f"sopra la nota {sopra:+.2f} Hz, sotto {sotto:+.2f} Hz")
print(f"oscillazioni complete in {durata} s: {fv * durata:.1f}")

# Controllo: rimessa in cent, l'escursione massima torna A.
print(f"escursione massima in cent: {1200 * np.log2(f0_tau / fm).max():.1f}")

### Dalla partitura al mel, con una rete


In [ ]:
import numpy as np

FPS = 24000 / 128          # frame al secondo del mel (24 kHz, hop 128)

# Tre note (inizio e durata in secondi) con le sillabe a-mo-re. Di ogni
# sillaba: i fonemi, e per le consonanti la durata in secondi.
note = [(0.00, 0.64), (0.64, 0.64), (1.28, 1.28)]
sillabe = [[("a", None)],
           [("m", 0.08), ("o", None)],
           [("r", 0.064), ("e", None)]]

# L'attacco della vocale cade sull'inizio della nota: la consonante parte
# prima e si prende la coda della nota precedente.
fonemi, confini = [], []
for (inizio, _), sillaba in zip(note, sillabe):
    istante = inizio - sum(d for _, d in sillaba if d)
    for nome, d in sillaba:
        fonemi.append(nome)
        confini.append(istante)
        istante += d or 0.0           # la vocale arriva al confine dopo
confini.append(note[-1][0] + note[-1][1])

frame = np.rint(np.array(confini) * FPS).astype(int)
durate = np.diff(frame)               # frame di ogni fonema
print(f"{FPS} frame al secondo")
for f, a, d in zip(fonemi, frame[:-1], durate):
    print(f"{f:>2}  dal frame {a:3d}  durata {d:3d} frame")

# Regolatore di lunghezza: ogni fonema si ripete per i suoi frame.
indici = np.repeat(np.arange(len(fonemi)), durate)
print("frame totali:", len(indici), "(la partitura dura", frame[-1], "frame)")

# Durata di ogni fonema come frazione della nota che porta la sillaba.
lunga = [int(round(d * FPS)) for _, d in note]
nota_di = [0, 1, 1, 2, 2]
rapporti = [round(float(d) / lunga[j], 3) for d, j in zip(durate, nota_di)]
print("fonema/nota:", rapporti)

# La somma dei rapporti, nota per nota (la consonante conta nella sua sillaba).
somme = [round(sum(r for r, j in zip(rapporti, nota_di) if j == k), 3)
         for k in range(len(note))]
print("somma per nota:", somme)

### Partire da un mel sfocato: la diffusione poco profonda


In [ ]:
import numpy as np

# Calendario lineare di DiffSinger: T = 100 passi, beta da 1e-4 a 0,06.
T, k = 100, 54
beta = np.linspace(1e-4, 0.06, T)
alfa_barra = np.cumprod(1.0 - beta)   # alfa_barra[t-1] e' \bar{alpha}_t
snr = alfa_barra / (1.0 - alfa_barra)

for t in (10, 30, k, T):
    print(f"t = {t:3d}   alfa_barra = {alfa_barra[t-1]:.4f}"
          f"   SNR = {snr[t-1]:.4f}")

# Una colonna di mel "vera" (80 bande) e la stessa colonna sfocata da una
# media mobile a tre punti. Sono dati di esempio; i bordi restano fuori.
bande = np.arange(80)
x0 = 0.5 * np.sin(bande / 3.0) + 0.3 * np.sin(bande * 1.3)
sfocata = np.convolve(x0, [0.25, 0.5, 0.25], mode="same")
x0, sfocata = x0[1:-1], sfocata[1:-1]
d2 = np.sum((x0 - sfocata) ** 2)
print(f"distanza al quadrato fra le due colonne: {d2:.3f}")

# Al passo t le due colonne diventano due gaussiane con la stessa varianza e
# medie che differiscono di sqrt(alfa_barra) * (x0 - sfocata): la loro KL vale
# 1/2 * SNR(t) * d2.
kl = 0.5 * snr * d2
for t in (10, 30, k, T):
    print(f"t = {t:3d}   KL = {kl[t-1]:.3f} nat")

# Criterio dell'appendice: il primo passo in cui la KL non supera quella fra
# q(x_T | x0) e N(0, I), che vale (a (|x0|^2 - dim) - dim log(1 - a)) / 2 con
# a = alfa_barra al passo T e dim componenti.
a, dim = alfa_barra[-1], len(x0)
soglia = 0.5 * (a * (np.sum(x0 ** 2) - dim) - dim * np.log(1.0 - a))
primo = int(np.argmax(kl <= soglia)) + 1
print(f"KL fra x_T e il rumore puro: {soglia:.3f} nat")
print(f"primo passo con KL non superiore: t = {primo}")

In [ ]:
# Valori dichiarati dagli autori di DiffSinger: T = 100 passi, k = 54, e il
# fattore di tempo reale (secondi di calcolo per secondo di audio).
T, k = 100, 54
rtf_completo, rtf_poco_profonda = 0.348, 0.191

print(f"passi: {k}/{T} = {k / T:.3f}")
rapporto = rtf_poco_profonda / rtf_completo
print(f"tempo: {rtf_poco_profonda}/{rtf_completo} = {rapporto:.3f}")
print(f"risparmio di tempo: {1 - rapporto:.1%}")

## La canzone come linguaggio: voce e accompagnamento in fila

[Leggi la pagina](https://book.paithon.it/main/GenerazioneMusica/la-canzone-come-linguaggio.html)


### Imparare l'allineamento: Jukebox


In [ ]:
import numpy as np

r = 50                                    # frame al secondo
sillabe = ["la", "lu", "na", "sul", "ma", "re"]
L = len(sillabe)
rng = np.random.default_rng(3)

veloce = rng.integers(8, 16, size=L)      # frame per sillaba
lenta = veloce.copy()
lenta[2] += 60                            # una vocale tenuta
lenta[5] += 90                            # e un'altra, in chiusura

for nome, d in [("veloce", veloce), ("vocali tenute", lenta)]:
    n = int(d.sum())                      # frame della riga
    a = np.concatenate(([0], np.cumsum(d)[:-1]))   # frame d'inizio a_j, da 0
    sillaba = np.repeat(np.arange(1, L + 1), d)       # sillaba di ogni frame
    lineare = np.arange(L) * n / L        # sillabe spalmate in modo uniforme
    errore = np.abs(lineare - a).max()
    print(f"{nome:14s} {n:3d} frame, {n / r:.2f} s, a = {a.tolist()}")
    print("  sillaba ogni 4 frame:", "".join(str(j) for j in sillaba[::4]))
    print(f"  stima lineare: errore massimo {errore:.1f} frame "
          f"({errore / r:.2f} s)")

# il testo scrive il ritornello una volta; il brano lo canta due volte
testo = ["strofa 1", "ritornello", "strofa 2"]
cantato = np.array([0, 1, 2, 1])          # indice della parte di testo
print("ordine cantato:", [testo[k] for k in cantato])
print("sempre in avanti nel testo?", bool(np.all(np.diff(cantato) > 0)))

### Voce e accompagnamento: due token per istante


In [ ]:
import numpy as np

n = 6                                          # frame
voce = np.array([512, 512, 87, 87, 301, 640])  # token inventati, traccia voce
acc = np.array([14, 933, 933, 21, 21, 77])     # traccia accompagnamento

fila = np.empty(2 * n, dtype=int)
fila[0::2] = voce                              # voce e acc., frame per frame
fila[1::2] = acc
print("fila intrecciata:", fila.tolist())

voce2, acc2 = fila[0::2], fila[1::2]           # la separazione inversa
print("separate di nuovo:", np.array_equal(voce2, voce),
      np.array_equal(acc2, acc))

# che cosa vede ogni token quando viene previsto: i posti che lo precedono
traccia = np.array(["V", "A"] * n)             # di chi è ogni posto della fila


def visti(posto):
    prima = traccia[:posto]
    return int(np.sum(prima == "V")), int(np.sum(prima == "A"))


print("frame | token voce vede (voce, acc) | token acc vede (voce, acc)")
for i in range(n):
    vv, va = visti(2 * i)                      # il posto della voce del frame
    av, aa = visti(2 * i + 1)                  # il posto dell'accompagnamento
    print(f"{i:5d} | {vv:12d} {va:5d}          | {av:12d} {aa:5d}")

bit = np.log2(1024)                            # codebook da 1024 segni
print(f"un token: {bit:.0f} bit; una traccia a 50 Hz: {bit * 50:.0f} bit/s; "
      f"due tracce: {2 * bit * 50:.0f} bit/s")
print(f"token semantici di AudioLM (1024 segni, 25 Hz): {bit * 25:.0f} bit/s")

### La struttura scritta sezione per sezione


In [ ]:
r = 50                                         # frame al secondo del codec
for nome, tracce in [("mix", 1), ("due tracce", 2)]:
    tok_s = r * tracce
    righe = [f"{k} token = {k / tok_s:.1f} s"
             for k in (3000, 6000, 8192, 16384)]
    print(f"{nome} ({tok_s} token/s):")
    print("  " + "; ".join(righe[:2]))
    print("  " + "; ".join(righe[2:]))

D, sezioni = 180, 14                           # sezioni di uguale durata
tok = D * 2 * r                                # token a due tracce
sez = tok / sezioni
print(f"{D} s a due tracce: {tok} token")
print(f"una sezione: {D / sezioni:.1f} s, {sez:.0f} token")
print(f"solo testo in testa: l'ultimo token dista {tok} token dal testo")
print(f"  oltre 3000 token: {(tok - 3000) / tok:.1%} della canzone")
print(f"  oltre 6000 token: {(tok - 6000) / tok:.1%} della canzone")
print(f"testo di sezione: al più {sez:.0f} token di audio dal proprio testo")

### Quanti passi in fila


In [ ]:
r, D, q = 50, 180, 4                           # Hz, secondi, livelli MusicGen
n = r * D                                      # frame in tre minuti
yue = 2 * n                                    # YuE: due token per frame
musicgen = n + q - 1                           # MusicGen con il delay pattern
q_yue = 8                                      # i livelli di X-Codec in YuE
ritardo_yue = n + q_yue - 1                    # il ritardo su quegli 8 livelli
contesto = 16384                               # il contesto più lungo di YuE
print(f"frame in {D} s a {r} Hz: {n}")
print(f"YuE, stadio 1 (due token per frame): {yue} passi in fila")
print(f"MusicGen (delay, q = {q}): {musicgen} passi in fila")
print(f"rapporto: {yue / musicgen:.2f}")
print(f"ritardo con q = {q_yue}: {ritardo_yue} passi, {ritardo_yue - n} più "
      f"di un token per frame ({(ritardo_yue - n) / n:.2%})")
print(f"contesto di {contesto} token = {contesto / (2 * r):.1f} s a due tracce")
print(f"{D} s entrano nel contesto? {yue <= contesto}")

cinque = 5 * 60 * 2 * r                        # cinque minuti a due tracce
vista = contesto - 3000 - 1                    # meno i token nuovi, meno 1
print(f"cinque minuti a due tracce: {cinque} token")
print(f"ultimi token tenuti in inferenza: {vista} = {vista / (2 * r):.1f} s")

## Tutta la canzone insieme: la diffusione su un latente

[Leggi la pagina](https://book.paithon.it/main/GenerazioneMusica/tutta-la-canzone-insieme.html)


In [ ]:
# Quanti passi in fila servono per tre minuti di canzone.
D = 180            # durata in secondi

# strada a token: un passo per token, in fila
yue = D * 50 * 2           # YuE: primo codebook a 50 Hz, due token per frame
n, q = D * 50, 4           # MusicGen: frame a 50 Hz, q livelli del codec
musicgen = n + q - 1       # schema a ritardo: n + q - 1 passi

# diffusione: ogni passo rifa tutta la canzone
passi = {"DiffRhythm": 32, "ACE-Step v1, 27 passi": 27, "ACE-Step v1, 60 passi": 60}

print(f"YuE, voce e accompagnamento   {yue:>6} passi in fila")
print(f"MusicGen, schema a ritardo    {musicgen:>6} passi in fila")
for nome, p in passi.items():
    print(f"{nome:<29} {p:>6} passi, ciascuno su tutta la canzone")

### Il latente: comprimere la canzone e poi rifarla


In [ ]:
# Dai numeri dell'onda ai numeri che il DiT vede, per quattro minuti.
D, fs = 240, 44100

onda = 2 * fs * D                  # stereo
mel_fps = fs / 512                 # frame del mel al secondo (hop 512)
mel = 2 * 128 * mel_fps * D        # due canali, 128 bande

lat_fps = fs / 512 / 8             # ACE-Step v1: tempo / 8
v1 = 8 * 16 * lat_fps * D          # 8 canali, 128 bande / 8 = 16 righe
dr_fps = fs / 2048                 # DiffRhythm: fattore 2048 nel tempo
dr = 64 * dr_fps * D               # 64 canali

print(f"onda stereo a 44100 Hz   {onda:>9.0f} numeri")
print(f"mel, due canali          {mel:>9.0f}   {onda/mel:.0f}x meno dell'onda")
print(f"latente ACE-Step v1      {v1:>9.0f}   {onda/v1:.0f}x meno dell'onda, {mel/v1:.0f}x meno del mel")
print(f"latente DiffRhythm       {dr:>9.0f}   {onda/dr:.0f}x meno dell'onda")
print(f"frame del latente v1     {lat_fps*D:>9.0f}   uno ogni {1000/lat_fps:.1f} ms")
print(f"frame del latente DiffR. {dr_fps*D:>9.0f}   uno ogni {1000/dr_fps:.1f} ms")

### Attenzione lineare sul tempo, testo per cross-attention


In [ ]:
# Costo dell'attenzione in un blocco del DiT: moltiplicazioni-somme per testa.
n, dk = 2584, 128                  # frame del latente (4 minuti), larghezza di una testa

softmax = 2 * n * n * dk           # QK^T poi A V: n x n punteggi
lineare = 2 * n * dk * dk          # stato S = somma v phi(k)^T, poi lettura
print(f"coppie di istanti (n x n)   {n*n:>11}")
print(f"auto-attenzione softmax     {softmax:>11}")
print(f"auto-attenzione lineare     {lineare:>11}   rapporto n/dk = {n/dk:.1f}")

# la cross-attention resta softmax: n istanti guardano nc token di condizione
for nc in (64, 512, 1 + 256 + 4096):   # l'ultimo e' il tetto: parlante, tag, testo
    cross = 2 * n * nc * dk
    print(f"cross-attention, nc = {nc:>4} {cross:>11}   {cross/softmax:.2f} volte la softmax su n")

In [ ]:
import numpy as np

# 1) Due token, chiavi e query di due numeri gia' passate dalla ReLU.
v = np.array([10.0, 20.0])                    # i valori dei due token
phi_k = np.array([[3.0, 0.0], [1.0, 2.0]])    # phi(k_0), phi(k_1)
S = v @ phi_k                                 # stato: somma di v_j phi(k_j)
z = phi_k.sum(axis=0)                         # normalizzatore: somma di phi(k_j)
print("stato S:", S, "  normalizzatore z:", z)
for phi_q in ([1.0, 0.0], [0.0, 1.0], [1.0, 1.0]):
    phi_q = np.array(phi_q)
    lettura = (S @ phi_q) / (z @ phi_q)       # una sola lettura per query
    pesi = phi_k @ phi_q                      # i pesi delle coppie, per confronto
    media = pesi @ v / pesi.sum()
    print(f"phi(q) = {phi_q}: lettura {lettura:5.2f}, "
          f"media pesata a coppie {media:5.2f}, pesi {pesi}")

# 2) RoPE e poi ReLU: il peso dipende ancora dalla sola distanza?
def ruota(x, pos, base=1e6):                  # rope_theta di ACE-Step v1
    ang = pos * base ** (-np.arange(0, len(x), 2) / len(x))
    c, s = np.cos(ang), np.sin(ang)
    y = x.copy()
    y[0::2] = c * x[0::2] - s * x[1::2]
    y[1::2] = s * x[0::2] + c * x[1::2]
    return y

rng = np.random.default_rng(4)
q, k = rng.standard_normal(8), rng.standard_normal(8)
relu = lambda x: np.maximum(0, x)
print("posizioni   senza ReLU   ReLU dopo la rotazione")
for i in (0, 10, 100):
    j = i + 3                                 # sempre la stessa distanza
    nudo = ruota(q, i) @ ruota(k, j)
    con_relu = relu(ruota(q, i)) @ relu(ruota(k, j))
    print(f"{i:>3} e {j:>3}   {nudo:10.4f}   {con_relu:10.4f}")

In [ ]:
# Velocita' dichiarate nella model card di ACE-Step v1: secondi di audio
# prodotti per secondo di calcolo (RTF, piu' alto e' piu' veloce).
rtf = {"A100": (27.27, 12.27), "RTX 4090": (34.48, 15.63), "RTX 3090": (12.76, 6.48)}

print(f"rapporto fra i passi, 60 / 27: {60 / 27:.2f}")
print("scheda    rapporto fra   parte fissa, ms   frazione del tempo")
print("          le velocita'   per s di audio    coi passi, a 27")
for scheda, (r27, r60) in rtf.items():
    # tempo per secondo di audio = parte fissa + parte per passo * passi:
    # due velocita' fissano i due coefficienti, ma non controllano il modello
    per_passo = (1 / r60 - 1 / r27) / (60 - 27)
    fissa = 1 / r27 - 27 * per_passo      # secondi di calcolo per secondo di audio
    quota = 27 * per_passo * r27          # frazione del tempo che va coi passi
    print(f"{scheda:<9} {r27 / r60:12.2f}   {1000 * fissa:+15.2f}   {quota:17.0%}")

### Allineare le rappresentazioni: tenere il testo in rotta


In [ ]:
import numpy as np

rng = np.random.default_rng(0)

# 1) La perdita del report, scritta sul dato pulito, e' la perdita sulla
#    velocita' pesata per t^2 (t = coefficiente del rumore).
x0 = rng.standard_normal((6, 40))        # dato pulito
eps = rng.standard_normal((6, 40))       # rumore
t = rng.uniform(0.05, 0.95, size=(6, 1))
v = rng.standard_normal((6, 40))         # una "predizione" qualunque della rete

xt = (1 - t) * x0 + t * eps              # convenzione del libro
u = eps - x0                             # bersaglio: la velocita'
x0_pred = xt - t * v                     # come nel report: v * (-t) + xt

perdita_dato = ((x0_pred - x0) ** 2).sum(axis=1)
perdita_vel = (t[:, 0] ** 2) * ((v - u) ** 2).sum(axis=1)
print("le due perdite coincidono:", np.allclose(perdita_dato, perdita_vel))

# 2) La perdita REPA: 1 - coseno, dopo aver portato gli stati del DiT
#    alla frequenza dei frame di MERT (interpolazione lineare).
r_dit, r_mert, finestra = 10.77, 75, 5.0     # frame al secondo; secondi
n_dit, n_mert = round(r_dit * finestra), round(r_mert * finestra)
print("frame del DiT e di MERT nella stessa finestra:", n_dit, n_mert)

tt = np.linspace(0, 1, n_mert)
y = np.stack([np.sin(2 * np.pi * f * tt + p)
              for f, p in [(1, 0.0), (2, 1.0), (3, 2.0), (1.5, 0.5)]], axis=1)

def porta_alla_frequenza(h, n_out):
    xs = np.linspace(0, 1, len(h))
    xo = np.linspace(0, 1, n_out)
    return np.stack([np.interp(xo, xs, h[:, c]) for c in range(h.shape[1])], axis=1)

def uno_meno_coseno(a, b):
    a = a / np.linalg.norm(a, axis=1, keepdims=True)
    b = b / np.linalg.norm(b, axis=1, keepdims=True)
    return (1 - (a * b).sum(axis=1)).mean()

h_buono = porta_alla_frequenza(y[np.linspace(0, n_mert - 1, n_dit).astype(int)], n_mert)
h_a_caso = porta_alla_frequenza(rng.standard_normal((n_dit, 4)), n_mert)
print(f"1 - coseno, stati coerenti     {uno_meno_coseno(h_buono, y):.4f}")
print(f"1 - coseno, stati a caso       {uno_meno_coseno(h_a_caso, y):.4f}")
print(f"solo il coseno, coerenti       {1 - uno_meno_coseno(h_buono, y):.4f}")

### Modificare senza rigenerare: variazioni, repaint, FlowEdit


In [ ]:
import numpy as np

rng = np.random.default_rng(7)
n = 200_000                          # numeri estratti per ogni rumore
eps_src = rng.standard_normal(n)     # il rumore con cui e' nata la canzone
eps_nuovo = rng.standard_normal(n)   # un rumore nuovo

print("forza  cos   sin   varianza  correlazione  varianza della media pesata")
for eta in (0.2, 0.5, 0.8):
    angolo = eta * np.pi / 2
    nuovo = np.cos(angolo) * eps_src + np.sin(angolo) * eps_nuovo
    ingenuo = (1 - eta) * eps_src + eta * eps_nuovo
    corr = np.corrcoef(eps_src, nuovo)[0, 1]
    print(f"{eta:5.1f}  {np.cos(angolo):.3f} {np.sin(angolo):.3f}"
          f"  {nuovo.var():8.3f}  {corr:12.3f}  {ingenuo.var():27.3f}")

## Prima il progetto, poi il suono

[Leggi la pagina](https://book.paithon.it/main/GenerazioneMusica/prima-il-progetto.html)


### Il renderer cambia pelle: dal mel all'onda


In [ ]:
# Dal brano ai numeri che la rete vede: quattro minuti di canzone
D = 240                                   # durata in secondi

onda = 48000 * 2 * D                      # onda stereo a 48 kHz
latente = 64 * 25 * D                     # 64 canali a 25 frame al secondo
token_dit = int(12.5 * D)                 # dopo la patch di 2 frame: 12,5 al secondo
codici = 5 * D                            # i codici del pianificatore, 5 al secondo

print("onda stereo 48 kHz          ", f"{onda:>11,}".replace(",", "."))
print("latente 64 x 25 Hz          ", f"{latente:>11,}".replace(",", "."),
      f"  ({onda / latente:.0f}x meno numeri)")
print("passo nel tempo              ", f"{48000 // 25}x  (48000 campioni / 25 frame)")
print("token del DiT               ", f"{token_dit:>11,}".replace(",", "."))
print("codici del pianificatore    ", f"{codici:>11,}".replace(",", "."))

# Attenzione a finestra contro attenzione globale, in coppie token-token.
# Nel codice la finestra e' bidirezionale, |i-j| <= w: ogni token ne vede
# w prima e w dopo, salvo ai bordi della canzone.
n, w = token_dit, 128
globali = n * n
a_finestra = n * (2 * w + 1) - w * (w + 1)
print()
print(f"{'coppie in uno strato globale':<32}{globali:>11,}".replace(",", "."))
print(f"{'coppie in uno strato a finestra':<32}{a_finestra:>11,}".replace(",", "."))
print(f"{'rapporto':<32}{globali / a_finestra:>11.1f}")
print(f"{'finestra per parte, in secondi':<32}{w / 12.5:>11.2f}")

# Dieci minuti, la durata massima dichiarata: negli strati globali
# le coppie crescono con il quadrato della durata
n10 = int(12.5 * 600)
coppie10 = f"{n10 * n10:>11,}".replace(",", ".")
print(f"{'coppie globali a dieci minuti':<32}{coppie10}")

# GQA: 16 teste di query, 8 di chiavi e valori
print("teste di chiavi e valori / teste di query:", 8 / 16)

### Il pianificatore: un modello di linguaggio che scrive lo schizzo


In [ ]:
import math

# Quanta informazione al secondo porta ogni rappresentazione discreta
def bit_s(livelli, segni, hz):
    return livelli * math.log2(segni) * hz

segni_fsq = math.prod([8, 8, 8, 5, 5, 5])   # combinazioni dei sei valori FSQ
righe = [
    ("token semantici di AudioLM (25 Hz)", bit_s(1, 1024, 25)),
    ("MusicGen, 4 livelli da 2048 (50 Hz)", bit_s(4, 2048, 50)),
    ("X-Codec, 8 livelli da 1024 (50 Hz)", bit_s(8, 1024, 50)),
    (f"codici FSQ, 1 livello da {segni_fsq} (5 Hz)", bit_s(1, segni_fsq, 5)),
]
for nome, b in righe:
    print(f"{nome:<42}{b:>7.1f} bit/s")
print("bit per codice FSQ:", round(math.log2(segni_fsq), 2),
      "  (con 16 bit:", 2**16, "combinazioni)")
print("X-Codec / codici FSQ:", round(righe[2][1] / righe[3][1]))

# Passi in fila per una canzone di tre minuti
D = 180
print()
print("YuE, stadio 1 (voce e accompagnamento):", 50 * 2 * D)
print("MusicGen, con il delay pattern:         ", 50 * D + 4 - 1)
print("pianificatore, codici a 5 Hz:           ", 5 * D)

### Distillare il renderer, rinforzare i due modelli


In [ ]:
# Il renderer: passi di diffusione, ciascuno su tutta la canzone
base_passi, turbo_passi = 50, 8
base_valutazioni = base_passi * 2      # con la guida: con e senza richiesta
turbo_valutazioni = turbo_passi * 1    # senza guida: una sola
print("DiT base: ", base_passi, "passi,", base_valutazioni, "valutazioni della rete")
print("DiT turbo:", turbo_passi, "passi,", turbo_valutazioni, "valutazioni della rete")
print("valutazioni in meno:", base_valutazioni / turbo_valutazioni, "volte")

In [ ]:
import numpy as np

# Una mappa di cross-attention in miniatura: 6 token del testo, 24 frame.
# Ogni frame distribuisce la propria attenzione sui token (softmax sui token).
n_tok, n_fr = 6, 24
frame = np.arange(n_fr)

def mappa(centri, spento=None):
    logit = -((frame[None, :] - np.asarray(centri)[:, None]) ** 2) / (2 * 2.0**2)
    if spento is not None:
        logit[spento] = -50.0              # nessun frame guarda questo token
    e = np.exp(logit - logit.max(axis=0))
    return e / e.sum(axis=0)

def punteggi(A):
    massa = A.sum(axis=1)                          # attenzione totale ricevuta da ogni token
    copertura = np.mean(massa >= 0.5)              # quota di token davvero letti
    centroidi = (A * frame).sum(axis=1) / massa
    monotonia = np.mean(np.diff(centroidi) > 0)    # quota di coppie in ordine
    return copertura, monotonia

ordinati = np.array([2, 6, 10, 14, 18, 22])
casi = {
    "ordinata": mappa(ordinati),
    "un token saltato": mappa(ordinati, spento=3),
    "due token scambiati": mappa(ordinati[[0, 3, 2, 1, 4, 5]]),
}
print(f"{'mappa':<22}{'copertura':>10}{'monotonia':>11}")
for nome, A in casi.items():
    c, m = punteggi(A)
    print(f"{nome:<22}{c:>10.2f}{m:>11.2f}")

### Abbozzare a patch alterne, suonare in tempo reale


In [ ]:
import math

# SpectroStream, il codec di Magenta RealTime: 25 frame al secondo,
# livelli da 1024 segni; in tempo reale si generano solo i primi 16
hz, segni = 25, 1024
for livelli in (64, 16):
    token = livelli * hz
    bit = token * math.log2(segni)
    print(f"{livelli} livelli: {token} token/s, {bit:.0f} bit/s")

### Misurare una canzone


In [ ]:
import numpy as np

rng = np.random.default_rng(0)

# Quattro canzoni e le loro quattro didascalie, già portate nello stesso spazio
# a 8 dimensioni: ogni coppia condivide una direzione, più un po' di rumore.
n, d = 4, 8
base = rng.normal(size=(n, d))
audio = base + 0.5 * rng.normal(size=(n, d))
testo = base + 0.5 * rng.normal(size=(n, d))

def unitari(x):
    return x / np.linalg.norm(x, axis=1, keepdims=True)

def perdita(testi, brani, tau=0.1):
    # la perdita contrastiva dal testo all'audio: log-softmax della coppia vera
    S = unitari(testi) @ unitari(brani).T
    logp = S / tau - np.log(np.exp(S / tau).sum(axis=1, keepdims=True))
    return float(-np.diag(logp).mean())

S = unitari(testo) @ unitari(audio).T          # coseno di ogni testo con ogni audio
print("coseno testo-audio (righe: testi, colonne: brani)")
print(np.round(S, 2))
print("coppie vere (diagonale), media:", round(float(np.diag(S).mean()), 2))
fuori = S[~np.eye(n, dtype=bool)]
print("coppie sbagliate, media:       ", round(float(fuori.mean()), 2),
      f"(da {fuori.min():.2f} a {fuori.max():.2f})")

# la stessa perdita con quattro didascalie a caso, senza legame con i brani
a_caso = rng.normal(size=(n, d))
print("perdita contrastiva, didascalie vere:  ", round(perdita(testo, audio), 3))
print("perdita contrastiva, didascalie a caso:", round(perdita(a_caso, audio), 3))
print("tirare a indovinare fra 4, log 4:      ", round(float(np.log(n)), 3))